# <center> <font color="#0036a3">Maestría en Inteligencia Artificial Aplicada (MNA)</font> </center>

<center>

[![Materia](https://img.shields.io/badge/MATERIA-PROYECTO_INTEGRADOR-E0A800?style=for-the-badge&logoColor=white)](https://tec.mx)

</center>

<center>

[![Python](https://img.shields.io/badge/Python-3776AB?style=flat-square&logo=python&logoColor=white)](https://www.python.org/)
[![Jupyter](https://img.shields.io/badge/Jupyter-F37626?style=flat-square&logo=jupyter&logoColor=white)](https://jupyter.org/)
[![PyTorch](https://img.shields.io/badge/PyTorch-EE4C2C?style=flat-square&logo=pytorch&logoColor=white)](https://pytorch.org/)
[![OpenCV](https://img.shields.io/badge/OpenCV-5C3EE8?style=flat-square&logo=opencv&logoColor=white)](https://opencv.org/)
[![GitHub](https://img.shields.io/badge/Repo-GitHub-181717?style=flat-square&logo=github&logoColor=white)](https://github.com/jmtoral/proyecto_integrador_52)

</center>

## **<font color="#0036a3">Semana 7: Avance 3 — Baseline de Estimación de Profundidad</font>**

### **<font color="#E0A800">Proyecto Integrador — TC5035.10</font>**
**Materia:** Proyecto Integrador — TC5035.10
**Fecha Límite:** 24 de mayo de 2026
**Framework de Calidad:** CRISP-ML(Q)

---

## **<center> <font color="#0036a3">Equipo 52</font> </center>**

<table style="border-collapse:collapse; width:60%; margin:auto;">
  <tr>
    <td align="center" style="border:none; width:33%; padding:10px;">
      <img src="https://raw.githubusercontent.com/jmtoral/proyecto_integrador_52/main/reports/elda.jpg" width="80" style="border-radius:50%; object-fit:cover;"><br>
      <strong>Elda Morales</strong><br><small>A00449074</small>
    </td>
    <td align="center" style="border:none; width:33%; padding:10px;">
      <img src="https://raw.githubusercontent.com/jmtoral/proyecto_integrador_52/main/reports/mpgc.jpg" width="80" style="border-radius:50%; object-fit:cover;"><br>
      <strong>María Paula Gutiérrez</strong><br><small>A01747706</small>
    </td>
    <td align="center" style="border:none; width:33%; padding:10px;">
      <img src="https://raw.githubusercontent.com/jmtoral/proyecto_integrador_52/main/reports/jmtc_n.jpg" width="80" style="border-radius:50%; object-fit:cover;"><br>
      <strong>José Manuel Toral</strong><br><small>A01122243</small>
    </td>
  </tr>
</table>

---

### **<font color="#E0A800">Información de Entrega</font>**
* **Actividad:** Avance 3. Baseline.
* **Fecha Límite:** 24 de mayo de 2026
* **Framework de Calidad:** CRISP-ML(Q)
* **Objetivos Clave:**
  * Establecer el **baseline de estimación de profundidad monocular** con Endo-Depth sobre SCARED (datasets 8–9), sin corrección de iluminación.
  * Evaluar el impacto de **CLAHE** y **Retinex SSR** sobre la calidad geométrica de la reconstrucción, usando AbsRel como métrica primaria.
  * Cuantificar si la mejora se concentra en **zonas especulares** (hipótesis central del proyecto).
  * Verificar la **viabilidad clínica** de cada corrección midiendo el costo computacional total respecto al umbral de 25 FPS.

---

---
## 0. Infraestructura de ejecución

Este notebook se ejecuta en **Google Colab** con la siguiente configuración de runtime:

### 0.1 Configuración de Colab

| Componente | Valor seleccionado |
|---|---|
| **Acelerador** | GPU A100 (opción de pago) |
| **RAM** | High RAM (~52 GB) |
| **Python** | 3.12.13 |
| **PyTorch** | 2.10.0+cu128 |
| **CUDA** | 12.8 |

Para reproducir: `Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU A100 + High RAM`.

### 0.2 GPU asignada: NVIDIA A100-SXM4-80GB

| Especificación | Valor |
|---|---|
| **Arquitectura** | Ampere (SM 8.0) |
| **VRAM** | 80 GB HBM2e |
| **Ancho de banda de memoria** | 2,039 GB/s |
| **Núcleos CUDA** | 6,912 |
| **Tensor Cores** | 432 (3ª gen, soporte BF16/TF32/FP16) |
| **NVLink** | Sí (SXM4 — interconexión de alta velocidad) |

Para este proyecto, Endo-Depth (ResNet18 + DepthDecoder) ocupa ~50 MB de VRAM — menos del 0.1% de los 80 GB disponibles. La A100 está sobredimensionada para inferencia con un modelo de este tamaño, pero garantiza que la Chamfer Distance (que construye KDTrees de ~900K puntos) y el loop completo de 10 keyframes × 3 correcciones terminen en minutos sin riesgo de OOM.

### 0.3 Acceso a datos y código

Todos los recursos necesarios residen en **Google Drive** y se montan en `/content/drive/MyDrive/`:

| Recurso | Ruta en Drive | Descripción |
|---|---|---|
| `scared_raw/` | `MyDrive/scared_raw/` | Datos SCARED (~250 GB en zips) |
| `endo_depth_weights/` | `MyDrive/endo_depth_weights/` | Pesos preentrenados (`encoder.pth`, `depth.pth`) |
| `Endo-Depth-and-Motion/` | `MyDrive/Endo-Depth-and-Motion/` | Código fuente del modelo (clonado manualmente una vez) |
| `avance3_outputs/` | `MyDrive/avance3_outputs/` | CSVs y PNGs de resultados (se crea automáticamente) |

### 0.4 Dependencias

Todas disponibles en el entorno `laluzqueopaca` (local) y en el runtime de Colab por defecto o con `pip install`:

```bash
pip install tifffile scikit-image
# torch, torchvision, opencv, scipy, pandas, matplotlib, tqdm ya están en Colab
```

### 0.5 Verificación del entorno

La celda siguiente confirma GPU, CUDA y versión de PyTorch antes de continuar.

In [ ]:
import torch, sys

print(f"Python  : {sys.version.split()[0]}")
print(f"PyTorch : {torch.__version__}")
print(f"CUDA disponible : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA versión    : {torch.version.cuda}")
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("Sin GPU — inferencia en CPU (correcto, solo más lento)")

---
## 1. Configuración de rutas y keyframes

In [ ]:
from pathlib import Path
import sys

# ── Detectar entorno ──────────────────────────────────────────────────────────
IN_COLAB = "google.colab" in sys.modules
if not IN_COLAB:
    try:
        import google.colab
        IN_COLAB = True
    except ImportError:
        IN_COLAB = False

if IN_COLAB:
    # ── Google Colab: montar Drive ────────────────────────────────────────────
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

    # El repo Endo-Depth-and-Motion ya está en Drive — no se clona.
    MODEL_PATH  = Path("/content/drive/MyDrive/endo_depth_weights")
    SCARED_ROOT = Path("/content/drive/MyDrive/scared_raw")
    EDAM_PATH   = Path("/content/drive/MyDrive/Endo-Depth-and-Motion")
    OUT_DIR     = Path("/content/drive/MyDrive/avance3_outputs")

else:
    # ── Local (VSCode + kernel laluzqueopaca) ─────────────────────────────────
    MODEL_PATH  = Path("E:/endo_depth_weights")
    SCARED_ROOT = Path("E:/scared_raw")
    EDAM_PATH   = Path("E:/Endo-Depth-and-Motion")
    OUT_DIR     = Path("../outcomes")

OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Keyframes de test (datasets 8–9 indexan desde 0) ─────────────────────────
# Para prueba rápida deja solo el primero.
# Para el experimento completo descomenta todos.
EVAL_KEYFRAMES = [
    ("dataset_8", "keyframe_0"),
    # ("dataset_8", "keyframe_1"),
    # ("dataset_8", "keyframe_2"),
    # ("dataset_8", "keyframe_3"),
    # ("dataset_8", "keyframe_4"),
    # ("dataset_9", "keyframe_0"),
    # ("dataset_9", "keyframe_1"),
    # ("dataset_9", "keyframe_2"),
    # ("dataset_9", "keyframe_3"),
    # ("dataset_9", "keyframe_4"),
]

CAP_MM = 150.0  # profundidad máxima a evaluar (igual que en Avance 2)

# ── Verificación ──────────────────────────────────────────────────────────────
print(f"Entorno  : {'Google Colab' if IN_COLAB else 'Local'}")
for name, p in [("MODEL_PATH",  MODEL_PATH),
                ("SCARED_ROOT", SCARED_ROOT),
                ("EDAM_PATH",   EDAM_PATH)]:
    status = "✓" if p.exists() else "✗ NO ENCONTRADA"
    print(f"  {name:13s}: {status}  ({p})")
print(f"  {'Keyframes':13s}: {len(EVAL_KEYFRAMES)} seleccionados")
print(f"  {'CAP_MM':13s}: {CAP_MM:.0f} mm")

---
## 2. Modelo Endo-Depth: arquitectura y funcionamiento

### 2.1 ¿Qué es Endo-Depth?

Endo-Depth es el estimador de profundidad monocular publicado por Recasens et al. (2021) como parte del sistema Endo-Depth-and-Motion. Está basado en **Monodepth2** (Godard et al., 2019) pero reentrenado sobre secuencias endoscópicas del dataset Hamlyn.

El modelo es **auto-supervisado**: durante el entrenamiento no usó mapas de profundidad GT, sino que aprendió a predecir profundidad minimizando el error de reconstrucción fotométrica entre frames consecutivos de video endoscópico.

### 2.2 Arquitectura

```
Entrada: imagen RGB (H×W×3)
    │
    ▼
ResNet18 Encoder
    ├── conv1   →  64 canales  (H/2  × W/2)
    ├── layer1  →  64 canales  (H/4  × W/4)
    ├── layer2  → 128 canales  (H/8  × W/8)
    ├── layer3  → 256 canales  (H/16 × W/16)
    └── layer4  → 512 canales  (H/32 × W/32)
    │
    ▼
DepthDecoder con skip connections
    ├── upconv4 (512→256) + skip layer3
    ├── upconv3 (256→128) + skip layer2  →  disp_3
    ├── upconv2 (128→64)  + skip layer1  →  disp_2
    ├── upconv1 (64→32)   + skip conv1   →  disp_1
    └── upconv0 (32→16)                  →  disp_0  ← salida principal
    │
    ▼
Sigmoid → disparidad en [0, 1]
    │
    ▼
disp_to_depth → profundidad relativa
```

### 2.3 De disparidad a profundidad

La red produce **disparidad** (no profundidad directamente). La conversión sigue la fórmula de Monodepth2:

$$\text{disp}_{\text{scaled}} = \frac{1}{d_{\max}} + \left(\frac{1}{d_{\min}} - \frac{1}{d_{\max}}\right) \cdot \text{disp}_{\text{sigmoid}}$$

$$\text{depth} = \frac{1}{\text{disp}_{\text{scaled}}}$$

donde $d_{\min} = 0.1$ y $d_{\max} = 100$ son los límites de profundidad durante el entrenamiento. El resultado está en **unidades relativas**, no en mm.

### 2.4 Escala métrica y median scaling

Los modelos monoculares auto-supervisados tienen una **ambigüedad de escala**: no pueden saber si la escena es pequeña y cercana o grande y lejana. Cuando hay GT disponible, se aplica **median scaling** para alinear la predicción con la escala real:

$$\hat{s} = \frac{\text{median}(d^*_{V})}{\text{median}(d_{V})}$$

$$d_{\text{metric}} = \hat{s} \cdot d_{\text{relativo}}$$

donde $V$ son los píxeles con GT válido. Este factor $\hat{s}$ varía por keyframe y **no es parte del modelo** — es un ajuste de evaluación estándar en la literatura.

In [ ]:
import sys
import torch
import numpy as np

sys.path.insert(0, str(EDAM_PATH / "apps" / "depth_estimate"))
from resnet_encoder import ResnetEncoder
from depth_decoder import DepthDecoder

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {DEVICE}")

encoder = ResnetEncoder(18, False)
loaded_enc = torch.load(MODEL_PATH / "encoder.pth", map_location=DEVICE)

FEED_HEIGHT = loaded_enc["height"]
FEED_WIDTH  = loaded_enc["width"]
print(f"Resolución de entrenamiento del modelo: {FEED_HEIGHT}×{FEED_WIDTH} px")

filtered_enc = {k: v for k, v in loaded_enc.items() if k in encoder.state_dict()}
encoder.load_state_dict(filtered_enc)
encoder.to(DEVICE).eval()

depth_decoder = DepthDecoder(num_ch_enc=encoder.num_ch_enc, scales=range(4))
loaded_dec = torch.load(MODEL_PATH / "depth.pth", map_location=DEVICE)
depth_decoder.load_state_dict(loaded_dec)
depth_decoder.to(DEVICE).eval()

n_params = sum(p.numel() for p in encoder.parameters()) + \
           sum(p.numel() for p in depth_decoder.parameters())
print(f"Parámetros totales del modelo: {n_params/1e6:.1f} M")
print("Modelo cargado ✓")

---
## 3. Carga de datos SCARED

### 3.1 Estructura del dataset

SCARED (Structured Light Endoscopy And Reconstructed Depth) provee pares imagen–profundidad capturados con luz estructurada sobre tejido porcino ex-vivo. Cada **keyframe** contiene:

| Archivo | Formato | Descripción |
|---|---|---|
| `Left_Image.png` | uint8 RGB 1280×1024 | Imagen de referencia — entrada al modelo |
| `left_depth_map.tiff` | float32 TIFF (H×W×3) | GT de profundidad: canales X, Y, **Z** en mm |

El canal **Z** del TIFF es la profundidad axial en milímetros. Los píxeles donde la luz estructurada no pudo medir (especulares, oclusiones, bordes) tienen valor ≤ 0 y se tratan como `NaN`.

### 3.2 Split del experimento

| Partición | Datasets | Animal | Keyframes | Propósito |
|---|---|---|---|---|
| **Train** | `dataset_1` – `dataset_7` | Cerdo A | kf_1 … kf_5 | Entrenamiento de los modelos supervisados |
| **Test** | `dataset_8`, `dataset_9` | Cerdo C | kf_0 … kf_4 | Evaluación — **este notebook** |

Este notebook evalúa solo sobre el **split de test**. El modelo Endo-Depth ya fue entrenado (no se reentrena aquí).

**¿Por qué datasets 8 y 9?** El split es por animal, no por keyframe. Los datasets 1–7 provienen de un individuo distinto al de los datasets 8–9. Usar animales diferentes para train y test evita que el modelo aprenda características idiosincrásicas del tejido de un cerdo concreto — textura, color, distribución de profundidad — y garantiza que los resultados miden generalización real, no memorización. Es el protocolo estándar del challenge SCARED (Allan et al., 2021).

> **Nota sobre los índices:** los keyframes de train (datasets 1–7) se indexan desde 1 (`keyframe_1` … `keyframe_5`); los de test (datasets 8–9) desde 0 (`keyframe_0` … `keyframe_4`). Es una inconsistencia del dataset original, no un error del código.

In [ ]:
import io, zipfile
import cv2
import tifffile
import numpy as np
import matplotlib.pyplot as plt


def load_keyframe_zip(zip_path, dataset_id, keyframe_id):
    with zipfile.ZipFile(zip_path) as z:
        buf = np.frombuffer(
            z.read(f"{dataset_id}/{keyframe_id}/Left_Image.png"), np.uint8)
        img = cv2.cvtColor(cv2.imdecode(buf, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
        with z.open(f"{dataset_id}/{keyframe_id}/left_depth_map.tiff") as f:
            tiff = tifffile.imread(io.BytesIO(f.read()))
        depth_z = tiff[..., 2].astype(np.float32)
        depth_z[depth_z <= 0] = np.nan
    return img, depth_z


def load_keyframe_dir(base_path, dataset_id, keyframe_id):
    kf = base_path / dataset_id / keyframe_id
    img = cv2.cvtColor(cv2.imread(str(kf / "Left_Image.png")), cv2.COLOR_BGR2RGB)
    tiff = tifffile.imread(str(kf / "left_depth_map.tiff"))
    depth_z = tiff[..., 2].astype(np.float32)
    depth_z[depth_z <= 0] = np.nan
    return img, depth_z


def load_keyframe(scared_root, dataset_id, keyframe_id):
    zip_path = scared_root / f"{dataset_id}.zip"
    if zip_path.exists():
        return load_keyframe_zip(zip_path, dataset_id, keyframe_id)
    return load_keyframe_dir(scared_root, dataset_id, keyframe_id)


# Carga y diagnóstico del primer keyframe
img_test, depth_test = load_keyframe(SCARED_ROOT, *EVAL_KEYFRAMES[0])
valid_mask = ~np.isnan(depth_test)

print(f"Imagen  : {img_test.shape}  dtype={img_test.dtype}")
print(f"Depth   : {depth_test.shape}  dtype={depth_test.dtype}")
print(f"Válidos : {valid_mask.mean()*100:.1f}%  "
      f"({valid_mask.sum():,} de {depth_test.size:,} píxeles)")
print(f"Rango Z : [{np.nanmin(depth_test):.1f}, {np.nanmax(depth_test):.1f}] mm")

# Visualización diagnóstica
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].imshow(img_test)
axes[0].set_title("Left_Image.png", fontsize=12)
axes[0].axis("off")

im = axes[1].imshow(depth_test, cmap="magma_r",
                    vmin=0, vmax=np.nanpercentile(depth_test, 98))
axes[1].set_title("GT profundidad Z (mm)", fontsize=12)
axes[1].axis("off")
plt.colorbar(im, ax=axes[1], label="mm")

axes[2].imshow(valid_mask, cmap="RdYlGn")
axes[2].set_title(f"Máscara GT válido ({valid_mask.mean()*100:.1f}%)", fontsize=12)
axes[2].axis("off")

plt.suptitle(f"{EVAL_KEYFRAMES[0]}", fontsize=13)
plt.tight_layout()
plt.show()

---
## 4. Correcciones de iluminación — variable independiente

Esta es la **variable independiente** del experimento factorial. Cada corrección transforma la imagen RGB antes de pasarla al modelo de profundidad. Todo lo demás (modelo, hiperparámetros, GT) permanece constante.

### 4.1 Baseline: sin corrección

La imagen original sin ningún preprocesamiento. Sirve como referencia para cuantificar el efecto de las correcciones.

### 4.2 CLAHE (Contrast Limited Adaptive Histogram Equalization)

CLAHE opera sobre el **canal de luminancia** en el espacio de color CIE Lab:

1. Convertir RGB → Lab
2. Dividir el canal L en tiles de tamaño `tile_size` (defecto 8×8)
3. En cada tile, ecualizar el histograma pero limitando la amplificación de contraste al valor `clip_limit` — los bins que superen ese límite redistribuyen su exceso al resto del histograma (esto evita amplificar ruido)
4. Interpolar bilinealmente entre tiles para evitar bordes artificiales
5. Convertir Lab → RGB

**¿Por qué funciona para endoscopía?** El endoscopio ilumina desde el centro del campo visual, creando un gradiente radial (brillante en el centro, oscuro en los bordes). CLAHE lo atenúa localmente sin perder información de textura.

### 4.3 Retinex (Single-Scale)

El modelo de Retinex (Land & McCann, 1971) postula que la imagen observada $I$ se puede descomponer como:

$$I(x,y) = L(x,y) \cdot R(x,y)$$

donde $L$ es la iluminación (variación lenta) y $R$ es la reflectancia (detalle real de la escena). En el dominio logarítmico:

$$\log I = \log L + \log R$$

$$\log R = \log I - \log L$$

Single-Scale Retinex estima $\log L$ mediante un filtro Gaussiano con desviación estándar $\sigma$:

$$R_{SSR}(x,y) = \log I(x,y) - \log \left[ G_\sigma * I(x,y) \right]$$

donde $G_\sigma$ es el kernel Gaussiano. Un $\sigma$ pequeño preserva detalles finos (bueno para texturas), un $\sigma$ grande elimina variaciones de iluminación más suaves.

**En endoscopía:** el gradiente radial de iluminación es la componente $L$, y Retinex lo elimina, dejando solo la reflectancia $R$ — idealmente más uniforme y con los reflejos especulares reducidos.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt


def correct_none(img_rgb: np.ndarray) -> np.ndarray:
    return img_rgb


def correct_clahe(img_rgb: np.ndarray,
                  clip_limit: float = 2.0,
                  tile_size: tuple = (8, 8)) -> np.ndarray:
    lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_size)
    lab[:, :, 0] = clahe.apply(lab[:, :, 0])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)


def correct_retinex(img_rgb: np.ndarray, sigma: float = 30) -> np.ndarray:
    img_f = img_rgb.astype(np.float32) + 1.0
    result = np.zeros_like(img_f)
    for c in range(3):
        blur = cv2.GaussianBlur(img_f[:, :, c], (0, 0), sigma)
        result[:, :, c] = np.log(img_f[:, :, c]) - np.log(blur + 1.0)
    result -= result.min()
    result = (result / (result.max() + 1e-8) * 255).astype(np.uint8)
    return result


CORRECTIONS = {
    "none":    correct_none,
    "clahe":   correct_clahe,
    "retinex": correct_retinex,
}

# Vista comparativa con histograma de luminancia
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
titles = {"none": "[A] Sin corrección (baseline)",
          "clahe": "[B] CLAHE (clip=2.0, tile=8×8)",
          "retinex": "[C] Retinex SSR (σ=30)"}

for col, (name, fn) in enumerate(CORRECTIONS.items()):
    corrected = fn(img_test)
    lum = cv2.cvtColor(corrected, cv2.COLOR_RGB2LAB)[:, :, 0]

    axes[0, col].imshow(corrected)
    axes[0, col].set_title(titles[name], fontsize=11)
    axes[0, col].axis("off")

    axes[1, col].hist(lum.ravel(), bins=100, color="steelblue", alpha=0.8)
    axes[1, col].set_xlabel("Luminancia (canal L)")
    axes[1, col].set_ylabel("Frecuencia")
    axes[1, col].set_title(f"μ={lum.mean():.1f}  σ={lum.std():.1f}")

plt.suptitle("Correcciones de iluminación y distribución de luminancia",
             fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

---
## 5. Inferencia con Endo-Depth

### 5.1 Pipeline de inferencia

Para cada imagen de entrada:

1. **Resize** a la resolución de entrenamiento del modelo (almacenada en `encoder.pth`). SCARED es 1280×1024; el modelo fue entrenado típicamente en 640×192 o similar para Hamlyn. El resize es necesario para que los pesos sean aplicables.

2. **Normalización implícita** en el encoder: el ResnetEncoder aplica internamente `(x - 0.45) / 0.225` antes de la primera capa conv — esta es la normalización ImageNet estándar.

3. **Forward pass**: encoder produce 5 feature maps multi-escala; el decoder los combina con skip connections para producir 4 disparidades a distintas escalas (`disp_0` a `disp_3`). Usamos `disp_0`, la de mayor resolución.

4. **Interpolación bilineal** de la disparidad de vuelta a la resolución original (1280×1024).

5. **Conversión disparidad → profundidad** con la fórmula de Monodepth2.

### 5.2 Medición de tiempo

El tiempo se mide con `time.perf_counter()` rodeando exactamente el forward pass (sin contar carga de imagen ni postprocesamiento). En GPU, `torch.cuda.synchronize()` es necesario antes de parar el cronómetro porque las operaciones CUDA son asíncronas por defecto.

In [ ]:
import time
import torch
import torch.nn.functional as F
import numpy as np
import PIL.Image as pil
from torchvision import transforms


def disp_to_depth(disp: np.ndarray,
                  min_depth: float = 0.1,
                  max_depth: float = 100.0) -> np.ndarray:
    """
    Convierte la disparidad sigmoidea del modelo a profundidad relativa.

    Fórmula (Monodepth2, Godard et al. 2019):
        disp_scaled = 1/d_max + (1/d_min - 1/d_max) * disp_sigmoid
        depth = 1 / disp_scaled
    """
    min_disp = 1.0 / max_depth
    max_disp = 1.0 / min_depth
    scaled_disp = min_disp + (max_disp - min_disp) * disp
    return 1.0 / scaled_disp


def predict_depth_timed(img_rgb: np.ndarray,
                        encoder, decoder,
                        feed_h: int, feed_w: int,
                        device) -> tuple:
    """
    Retorna (depth_relativo, tiempo_ms).
    El tiempo incluye solo el forward pass, no la carga de imagen.
    """
    H, W = img_rgb.shape[:2]

    # Preprocesamiento
    input_pil = pil.fromarray(img_rgb).resize((feed_w, feed_h), pil.LANCZOS)
    input_t = transforms.ToTensor()(input_pil).unsqueeze(0).to(device)

    # Forward pass cronometrado
    if device.type == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()

    with torch.no_grad():
        features = encoder(input_t)
        outputs  = decoder(features)

    if device.type == "cuda":
        torch.cuda.synchronize()
    elapsed_ms = (time.perf_counter() - t0) * 1000

    # Postprocesamiento
    disp = outputs[("disp", 0)]
    disp_full = F.interpolate(disp, (H, W), mode="bilinear", align_corners=False)
    disp_np = disp_full.squeeze().cpu().numpy()

    return disp_to_depth(disp_np), elapsed_ms


# Prueba rápida
depth_rel, t_ms = predict_depth_timed(
    img_test, encoder, depth_decoder, FEED_HEIGHT, FEED_WIDTH, DEVICE)
print(f"Resolución predicción : {depth_rel.shape}")
print(f"Rango (relativo)      : [{depth_rel.min():.3f}, {depth_rel.max():.3f}]")
print(f"Tiempo forward pass   : {t_ms:.1f} ms  ({1000/t_ms:.1f} FPS teórico)")

---
## 6. Métricas de evaluación — teoría completa

### 6.1 Métricas geométricas (vs. Ground Truth de profundidad)

Estas son las métricas principales del experimento. Todas comparan el **depth map predicho por Endo-Depth** contra el **GT de luz estructurada** de SCARED.

#### AbsRel (Absolute Relative Error) — métrica primaria

$$\text{AbsRel} = \frac{1}{|V|} \sum_{i \in V} \frac{|d_i - d_i^*|}{d_i^*}$$

donde $d_i$ es la profundidad predicha, $d_i^*$ es el GT, y $V$ es el conjunto de píxeles con GT válido dentro del rango de saturación. El denominador $d_i^*$ **normaliza por la profundidad real**, de modo que un error de 5 mm a 20 mm de distancia pesa igual que un error de 25 mm a 100 mm. Esto es importante en endoscopía donde el rango de profundidad es amplio (10–150 mm). Un AbsRel de 0.10 significa error relativo medio del 10%.

#### RMSE (Root Mean Square Error)

$$\text{RMSE} = \sqrt{\frac{1}{|V|} \sum_{i \in V} (d_i - d_i^*)^2}$$

A diferencia de AbsRel, RMSE está en **milímetros** y penaliza desproporcionadamente los errores grandes (por el cuadrado). Es útil para detectar predicciones catastróficas en zonas puntuales.

#### Chamfer Distance

La Chamfer Distance mide la similitud entre dos **nubes de puntos 3D** — no entre mapas 2D. Para convertir los depth maps a nubes de puntos necesitamos la **matriz intrínseca de la cámara** $K$:

$$\mathbf{p}_{3D} = \begin{pmatrix} (u - c_x) \cdot Z / f_x \\ (v - c_y) \cdot Z / f_y \\ Z \end{pmatrix}$$

donde $(u, v)$ son coordenadas de píxel, $(c_x, c_y)$ es el punto principal, $f_x, f_y$ son las longitudes focales, y $Z$ es la profundidad.

La Chamfer Distance entre dos nubes $P$ (predicción) y $Q$ (GT) es:

$$\text{CD}(P, Q) = \frac{1}{|P|} \sum_{p \in P} \min_{q \in Q} \|p - q\|_2 \;+\; \frac{1}{|Q|} \sum_{q \in Q} \min_{p \in P} \|q - p\|_2$$

**Implementación:** usamos `scipy.spatial.cKDTree` para los nearest neighbors — $O(N \log N)$ en lugar de $O(N^2)$ de fuerza bruta. Para imágenes 1280×1024 con ~70% de GT válido, la nube tiene ~900K puntos; el KDTree es esencial.

### 6.2 Métricas visuales (imagen corregida vs. imagen original) ⚠️

> **Importante:** PSNR y SSIM en este notebook **no miden el error de profundidad**. Comparan la imagen RGB corregida contra la imagen RGB original, para responder una pregunta distinta: *¿cuánto altera visualmente la imagen cada corrección?*
>
> Esto es relevante para viabilidad clínica: una corrección que mejora AbsRel pero produce una imagen irreconocible no sirve en quirófano — el cirujano necesita ver el tejido con claridad. PSNR y SSIM cuantifican ese trade-off.
>
> Para la condición `none` (sin corrección) estos valores son siempre **∞ y 1.0** porque la imagen de entrada y salida son idénticas. Ese resultado es matemáticamente correcto y se excluye del análisis comparativo — solo tiene sentido comparar PSNR/SSIM entre `clahe` y `retinex`.

#### PSNR (Peak Signal-to-Noise Ratio)

$$\text{MSE} = \frac{1}{H \cdot W \cdot C} \sum_{h,w,c} (I_{\text{orig}}(h,w,c) - I_{\text{corr}}(h,w,c))^2$$

$$\text{PSNR} = 10 \cdot \log_{10}\left(\frac{255^2}{\text{MSE}}\right)$$

Valores de referencia: > 40 dB cambio imperceptible · 30–40 dB cambio leve · 20–30 dB cambio notable · < 20 dB imagen muy alterada.

#### SSIM (Structural Similarity Index)

$$\text{SSIM}(x, y) = \frac{(2\mu_x\mu_y + C_1)(2\sigma_{xy} + C_2)}{(\mu_x^2 + \mu_y^2 + C_1)(\sigma_x^2 + \sigma_y^2 + C_2)}$$

Rango $[-1, 1]$, donde 1 = imágenes idénticas. Captura luminancia, contraste y estructura por separado — modela mejor la percepción visual humana que PSNR.

### 6.3 Métrica de robustez: AbsRel en zona especular (vs. GT)

Esta es la métrica **clave** para la hipótesis del proyecto. Compara el error de profundidad en dos zonas de la imagen:

1. **Detectar especulares** en la imagen original: píxeles con luminancia > percentil 97 en el canal L (Lab).
2. **Dilatar la máscara** con un kernel 15×15 para incluir el halo alrededor del especular.
3. **Calcular AbsRel vs. GT** separadamente en zona especular (`AbsRel_spec`) y el resto (`AbsRel_nospec`).

**Hipótesis:** si la corrección funciona, `AbsRel_spec` debería reducirse más que `AbsRel_nospec` — la corrección ayuda principalmente donde el modelo estaba confundido por sobreexposición.

In [ ]:
import numpy as np
import cv2
from scipy.spatial import cKDTree
from skimage.metrics import structural_similarity as ssim_fn
from skimage.metrics import peak_signal_noise_ratio as psnr_fn


# ── Calibración aproximada SCARED ────────────────────────────────────────────
# Valores del archivo endoscope_calibration.yaml (aproximados).
# Si tienes el yaml de tu keyframe específico, usa esos valores.
FX, FY = 1078.0, 1078.0   # longitudes focales en píxeles
CX, CY = 640.0,  512.0    # punto principal (centro de 1280×1024)


def depth_to_pointcloud(depth_mm: np.ndarray,
                        fx=FX, fy=FY, cx=CX, cy=CY,
                        mask: np.ndarray = None) -> np.ndarray:
    """
    Convierte un depth map (H×W) en mm a nube de puntos (N×3) en mm.
    mask: máscara booleana de píxeles a incluir (True = incluir).
    """
    H, W = depth_mm.shape
    u = np.arange(W)
    v = np.arange(H)
    uu, vv = np.meshgrid(u, v)

    valid = np.isfinite(depth_mm) & (depth_mm > 0)
    if mask is not None:
        valid &= mask

    Z = depth_mm[valid]
    X = (uu[valid] - cx) * Z / fx
    Y = (vv[valid] - cy) * Z / fy
    return np.stack([X, Y, Z], axis=1)


def chamfer_distance(pc_pred: np.ndarray, pc_gt: np.ndarray,
                     max_points: int = 50_000) -> float:
    """
    Chamfer Distance simétrica entre dos nubes de puntos 3D (en mm).
    Submuestrea si hay más de max_points para controlar tiempo de cómputo.
    """
    if len(pc_pred) == 0 or len(pc_gt) == 0:
        return np.nan

    # Submuestreo aleatorio si las nubes son muy grandes
    rng = np.random.default_rng(42)
    if len(pc_pred) > max_points:
        pc_pred = pc_pred[rng.choice(len(pc_pred), max_points, replace=False)]
    if len(pc_gt) > max_points:
        pc_gt = pc_gt[rng.choice(len(pc_gt), max_points, replace=False)]

    tree_gt   = cKDTree(pc_gt)
    tree_pred = cKDTree(pc_pred)

    dist_pred_to_gt, _ = tree_gt.query(pc_pred, k=1)
    dist_gt_to_pred, _ = tree_pred.query(pc_gt, k=1)

    return float((dist_pred_to_gt.mean() + dist_gt_to_pred.mean()) / 2)


def specular_mask(img_rgb: np.ndarray,
                  percentile: float = 97,
                  dilation_px: int = 15) -> np.ndarray:
    """
    Máscara booleana de zonas especulares (sobreexpuestas).
    Detecta píxeles con luminancia > percentile en canal L (Lab),
    luego dilata para incluir el halo circundante.
    """
    lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)
    L = lab[:, :, 0].astype(np.float32)
    threshold = np.percentile(L, percentile)
    mask = (L >= threshold).astype(np.uint8)
    kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE, (dilation_px, dilation_px))
    mask = cv2.dilate(mask, kernel)
    return mask.astype(bool)


def compute_all_metrics(img_orig: np.ndarray,
                        img_corrected: np.ndarray,
                        depth_pred_rel: np.ndarray,
                        gt_mm: np.ndarray,
                        cap_mm: float = 150.0) -> dict:
    """
    Calcula el conjunto completo de métricas para un par (corrección, keyframe).

    Parámetros
    ----------
    img_orig       : imagen RGB original (sin corrección), uint8
    img_corrected  : imagen RGB tras la corrección, uint8
    depth_pred_rel : depth map predicho en unidades relativas (H×W)
    gt_mm          : depth map GT en mm, NaN donde no hay medición
    cap_mm         : profundidad máxima a evaluar

    Retorna
    -------
    dict con todas las métricas
    """
    out = {}

    # ── Máscara de GT válido ──────────────────────────────────────────────
    valid = (~np.isnan(gt_mm)) & (gt_mm > 0) & (gt_mm < cap_mm)
    if valid.sum() == 0:
        return {k: np.nan for k in
                ["AbsRel","RMSE","Chamfer","PSNR","SSIM",
                 "AbsRel_spec","AbsRel_nospec","n_valid","scale"]}

    # ── Median scaling ────────────────────────────────────────────────────
    scale = np.median(gt_mm[valid]) / (np.median(depth_pred_rel[valid]) + 1e-8)
    pred_mm = depth_pred_rel * scale
    out["scale"] = round(float(scale), 4)
    out["n_valid"] = int(valid.sum())

    d  = pred_mm[valid]
    gt = gt_mm[valid]

    # ── AbsRel ───────────────────────────────────────────────────────────
    out["AbsRel"] = round(float(np.mean(np.abs(d - gt) / (gt + 1e-8))), 4)

    # ── RMSE ─────────────────────────────────────────────────────────────
    out["RMSE"] = round(float(np.sqrt(np.mean((d - gt) ** 2))), 2)

    # ── Chamfer Distance ──────────────────────────────────────────────────
    # Nube GT: solo píxeles válidos dentro del cap
    gt_clipped = np.where(valid, gt_mm, np.nan)
    pred_clipped = np.where(valid, pred_mm, np.nan)
    pc_gt   = depth_to_pointcloud(gt_clipped)
    pc_pred = depth_to_pointcloud(pred_clipped)
    out["Chamfer"] = round(chamfer_distance(pc_pred, pc_gt), 3)

    # ── PSNR ─────────────────────────────────────────────────────────────
    # Comparación imagen original vs. imagen corregida
    out["PSNR"] = round(float(psnr_fn(img_orig, img_corrected,
                                       data_range=255)), 2)

    # ── SSIM ─────────────────────────────────────────────────────────────
    out["SSIM"] = round(float(ssim_fn(img_orig, img_corrected,
                                       channel_axis=2, data_range=255)), 4)

    # ── Robustez: AbsRel en zona especular vs. resto ───────────────────
    spec_mask = specular_mask(img_orig)   # máscara sobre imagen ORIGINAL

    valid_spec   = valid & spec_mask
    valid_nospec = valid & ~spec_mask

    if valid_spec.sum() > 0:
        d_s = pred_mm[valid_spec]
        g_s = gt_mm[valid_spec]
        out["AbsRel_spec"] = round(
            float(np.mean(np.abs(d_s - g_s) / (g_s + 1e-8))), 4)
    else:
        out["AbsRel_spec"] = np.nan

    if valid_nospec.sum() > 0:
        d_ns = pred_mm[valid_nospec]
        g_ns = gt_mm[valid_nospec]
        out["AbsRel_nospec"] = round(
            float(np.mean(np.abs(d_ns - g_ns) / (g_ns + 1e-8))), 4)
    else:
        out["AbsRel_nospec"] = np.nan

    return out


print("Funciones de métricas definidas ✓")
print(f"\nMáscara especular de prueba: "
      f"{specular_mask(img_test).mean()*100:.1f}% del área de la imagen")

### 6.4 Visualización de la máscara especular

Antes de correr el experimento, verificamos que la máscara de especulares capture correctamente las zonas de sobreexposición.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

spec = specular_mask(img_test)

# Overlay de la máscara sobre la imagen
overlay = img_test.copy()
overlay[spec] = [255, 80, 80]   # rojo donde hay especular

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].imshow(img_test)
axes[0].set_title("Imagen original", fontsize=12)
axes[0].axis("off")

axes[1].imshow(spec, cmap="Reds")
axes[1].set_title(f"Máscara especular ({spec.mean()*100:.1f}% del área)",
                  fontsize=12)
axes[1].axis("off")

axes[2].imshow(overlay)
axes[2].set_title("Overlay (rojo = zona especular)", fontsize=12)
axes[2].axis("off")

plt.suptitle("Detección de zonas especulares (L > p97, dilatación 15px)",
             fontsize=13)
plt.tight_layout()
plt.show()

# ¿Cuántos píxeles con GT válido caen en zona especular?
valid_base = (~np.isnan(depth_test)) & (depth_test > 0) & (depth_test < CAP_MM)
print(f"Píxeles GT válidos en zona especular : "
      f"{(valid_base & spec).sum():,} "
      f"({(valid_base & spec).mean()*100:.1f}% del total válido)")
print(f"Píxeles GT válidos fuera de especular: "
      f"{(valid_base & ~spec).sum():,} "
      f"({(valid_base & ~spec).mean()*100:.1f}% del total válido)")

---
## 7. Experimento factorial: 3 correcciones × N keyframes

Este es el bucle principal. Para cada combinación (corrección, keyframe):

1. Cargar imagen RGB y GT de profundidad
2. Aplicar la corrección de iluminación
3. Inferir el depth map con Endo-Depth (cronometrado)
4. Calcular el conjunto completo de métricas
5. Guardar en la tabla de resultados

**Nota sobre el tiempo:** se hacen 3 runs por combinación y se reporta la mediana para reducir varianza por efectos del SO (interrupciones, caché).

In [ ]:
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm

N_TIMING_RUNS = 3   # repeticiones para estabilizar la medición de tiempo
results = []

for dataset_id, keyframe_id in tqdm(EVAL_KEYFRAMES, desc="Keyframes"):
    img_rgb, gt_mm = load_keyframe(SCARED_ROOT, dataset_id, keyframe_id)

    for corr_name, corr_fn in CORRECTIONS.items():
        img_corrected = corr_fn(img_rgb)

        # Múltiples runs para tiempo estable
        times_ms = []
        for _ in range(N_TIMING_RUNS):
            depth_rel, t_ms = predict_depth_timed(
                img_corrected, encoder, depth_decoder,
                FEED_HEIGHT, FEED_WIDTH, DEVICE)
            times_ms.append(t_ms)

        tiempo_ms = float(np.median(times_ms))
        fps = round(1000.0 / tiempo_ms, 2)

        # Todas las métricas
        metrics = compute_all_metrics(
            img_rgb, img_corrected, depth_rel, gt_mm, cap_mm=CAP_MM)

        results.append({
            "Dataset":       dataset_id,
            "Keyframe":      keyframe_id,
            "Corrección":    corr_name,
            "Modelo":        "EndoDepth (ResNet18)",
            "Tiempo_ms":     round(tiempo_ms, 1),
            "FPS":           fps,
            **metrics,
        })

        print(f"  {dataset_id}/{keyframe_id} | {corr_name:8s} | "
              f"AbsRel={metrics['AbsRel']:.4f} "
              f"RMSE={metrics['RMSE']:.1f}mm "
              f"CD={metrics['Chamfer']:.2f}mm "
              f"PSNR={metrics['PSNR']:.1f}dB "
              f"SSIM={metrics['SSIM']:.3f} "
              f"t={tiempo_ms:.0f}ms")

df = pd.DataFrame(results)
print("\nExperimento completo ✓")

---
## 8. Tabla de resultados y análisis

### 8.1 Resultados por keyframe

In [ ]:
import pandas as pd

cols_display = ["Dataset", "Keyframe", "Corrección",
                "AbsRel", "RMSE", "Chamfer",
                "PSNR", "SSIM",
                "AbsRel_spec", "AbsRel_nospec",
                "Tiempo_ms", "FPS"]

pd.set_option("display.float_format", "{:.4f}".format)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)
df[cols_display]

### 8.2 Promedio por corrección (agregado sobre todos los keyframes)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

numeric_cols = ["AbsRel", "RMSE", "Chamfer",
                "PSNR", "SSIM",
                "AbsRel_spec", "AbsRel_nospec",
                "Tiempo_ms", "FPS"]

summary = (
    df.groupby("Corrección")[numeric_cols]
    .mean()
    .round(4)
    .sort_values("AbsRel")
)

print("═" * 80)
print("RESUMEN — Promedio por corrección de iluminación")
print("═" * 80)
print(summary.to_string())
print("─" * 80)

winner_geo  = summary["AbsRel"].idxmin()
winner_vis  = summary["SSIM"].idxmax()
winner_spec = summary["AbsRel_spec"].idxmin()

print(f"\nMenor AbsRel global          : {winner_geo}")
print(f"Mayor SSIM (menos alteración): {winner_vis}")
print(f"Menor AbsRel en especulares  : {winner_spec}")

### 8.3 Visualización comparativa de métricas

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

corrections = list(summary.index)
x = np.arange(len(corrections))
colors = ["#2766CB", "#E0A800", "#2ecc71"]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle("Comparativa de métricas por corrección de iluminación",
             fontsize=14, fontweight="bold")

metrics_plot = [
    ("AbsRel",       "AbsRel (↓ mejor)",           "Geométrica"),
    ("RMSE",         "RMSE mm (↓ mejor)",           "Geométrica"),
    ("Chamfer",      "Chamfer Distance mm (↓ mejor)","Geométrica"),
    ("PSNR",         "PSNR dB (↑ mejor)",           "Visual"),
    ("SSIM",         "SSIM (↑ mejor)",              "Visual"),
    ("AbsRel_spec",  "AbsRel zona especular (↓ mejor)", "Robustez"),
]

for ax, (col, label, cat) in zip(axes.ravel(), metrics_plot):
    vals = [summary.loc[c, col] if c in summary.index else np.nan
            for c in corrections]
    bars = ax.bar(x, vals, color=colors[:len(corrections)],
                  edgecolor="white", linewidth=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(corrections, fontsize=11)
    ax.set_title(f"{label}\n[{cat}]", fontsize=10)
    ax.set_ylabel(label.split(" ")[0])
    for bar, val in zip(bars, vals):
        if not np.isnan(val):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                    f"{val:.3f}", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.savefig(OUT_DIR / "avance3_metricas_comparativa.png",
            dpi=150, bbox_inches="tight")
plt.show()

### 8.4 Análisis de robustez: especular vs. no especular

Si la hipótesis del proyecto es correcta, la mejora en AbsRel debería concentrarse en las zonas especulares. Este gráfico lo muestra explícitamente.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle("Robustez a iluminación: AbsRel en zona especular vs. resto",
             fontsize=13, fontweight="bold")

corr_list = list(summary.index)
x = np.arange(len(corr_list))
w = 0.35

for ax, kf_filter in zip(axes, [None, EVAL_KEYFRAMES[0]]):
    if kf_filter is None:
        sub = df
        title = "Promedio — todos los keyframes"
    else:
        sub = df[(df["Dataset"] == kf_filter[0]) &
                 (df["Keyframe"] == kf_filter[1])]
        title = f"{kf_filter[0]}/{kf_filter[1]}"

    spec_vals   = [sub[sub["Corrección"] == c]["AbsRel_spec"].mean()
                   for c in corr_list]
    nospec_vals = [sub[sub["Corrección"] == c]["AbsRel_nospec"].mean()
                   for c in corr_list]

    b1 = ax.bar(x - w/2, spec_vals,   w, label="Zona especular",
                color="#e74c3c", alpha=0.85)
    b2 = ax.bar(x + w/2, nospec_vals, w, label="Fuera de especular",
                color="#2766CB", alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(corr_list, fontsize=11)
    ax.set_ylabel("AbsRel")
    ax.set_title(title, fontsize=11)
    ax.legend(fontsize=9)

    for bar, val in [(b, v) for bars, vals in [(b1, spec_vals), (b2, nospec_vals)]
                     for b, v in zip(bars, vals) if not np.isnan(v)]:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                f"{val:.3f}", ha="center", va="bottom", fontsize=8)

plt.tight_layout()
plt.savefig(OUT_DIR / "avance3_robustez_especular.png",
            dpi=150, bbox_inches="tight")
plt.show()

# Interpretación cuantitativa
print("\n── Reducción de AbsRel_spec respecto al baseline (none) ──")
baseline_spec = summary.loc["none", "AbsRel_spec"] if "none" in summary.index else np.nan
for c in corr_list:
    if c == "none" or np.isnan(baseline_spec):
        continue
    v = summary.loc[c, "AbsRel_spec"]
    mejora = (baseline_spec - v) / baseline_spec * 100
    print(f"  {c:8s}: {v:.4f}  ({mejora:+.1f}% vs baseline)")

---
## 9. Visualización cualitativa

Mapa de profundidad predicho vs GT para cada corrección, con las métricas superpuestas. La escala de color `magma_r` mapea colores fríos a mayor profundidad (más lejano) y cálidos a menor profundidad (más cercano).

### 9.1 Interpretación de la visualización cualitativa (dataset_8 / keyframe_0)

La figura anterior tiene tres filas:

- **Fila 1 — imagen RGB** tras cada corrección (más la original como referencia).
- **Fila 2 — depth map predicho** por Endo-Depth, escalado a mm con median scaling. Color claro = cerca, oscuro = lejos.
- **Fila 3 — error absoluto por píxel** (|predicción − GT|). Amarillo/blanco = error alto, negro = error bajo. Gris = píxeles sin GT.

#### Lo que se observa

**Sin corrección (`none`):** el depth map de la fila 2 es casi completamente uniforme — Endo-Depth no detecta estructura geométrica relevante, produce una estimación plana que no corresponde a la topografía real del tejido. El error en la fila 3 es alto y distribuido por toda la imagen.

**CLAHE:** el depth map gana algo de contraste respecto al baseline, pero el error absoluto empeora (RMSE = 27.5 mm vs. 18.6 mm del baseline). La ecualización local introduce variaciones de textura que el modelo confunde con cambios de profundidad — lo que se ve como más estructura en el depth map en realidad es más ruido geométrico.

**Retinex:** produce el mejor resultado geométrico. El depth map (fila 2) captura más estructura de la escena, y el mapa de error (fila 3) es visualmente más oscuro — menos error distribuido. AbsRel = 0.084, RMSE = 14.0 mm, Chamfer = 5.64 mm, todos mínimos entre las tres condiciones. Sin embargo, la imagen RGB (fila 1) aparece grisácea y desaturada, lo que refleja el PSNR de 14.3 dB — la corrección altera significativamente la apariencia visual.

**Máscara especular (columna derecha, fila 3):** las manchas oscuras marcan las zonas de sobreexposición detectadas. Son precisamente las áreas donde el GT de luz estructurada falla (gris en la fila 3) — confirmando la correlación entre especulares y GT faltante documentada en el Avance 1.

#### Interpretación global (10 keyframes, datasets 8–9)

| | AbsRel | RMSE mm | Chamfer mm | PSNR imagen | Nota |
|---|---|---|---|---|---|
| **Retinex** | **0.153** | **11.58** | **5.66** | 12.4 dB | Mejor geometría, imagen muy alterada |
| Sin corrección | 0.165 | 12.47 | 5.79 | — | Referencia |
| CLAHE | 0.171 | 13.68 | 5.89 | 19.1 dB | Peor geometría, imagen más conservada |

**Retinex mejora el AbsRel un 7.2% respecto al baseline** — real pero modesto al promediar 10 keyframes. El margen era mayor (38.7%) sobre el keyframe_0 aislado porque ese keyframe tiene más reflejos especulares que el promedio de los datasets 8–9.

**La hipótesis se confirma parcialmente:** Retinex reduce `AbsRel_spec` de 0.209 → 0.195 (−7%), actuando principalmente donde se esperaba. El efecto es más pequeño al promediar porque varios keyframes de los datasets 8–9 tienen menos especularidad que el keyframe_0.

**CLAHE perjudica en todos los keyframes.** La ecualización adaptativa amplifica variaciones locales de textura que Endo-Depth interpreta como profundidad — un efecto adverso sistemático visible tanto en la visualización cualitativa como en todas las métricas geométricas.

**Trade-off clínico:** Retinex mejora la estimación de profundidad pero altera la imagen con PSNR ≈ 12–15 dB (imagen visualmente desaturada, como se ve en la fila 1). CLAHE conserva mejor la apariencia (PSNR ≈ 19 dB) pero empeora la profundidad. En un contexto quirúrgico habría que decidir qué priorizar: precisión geométrica para navegación o fidelidad visual para el cirujano.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

img_rgb, gt_mm = load_keyframe(SCARED_ROOT, *EVAL_KEYFRAMES[0])
vmax = np.nanpercentile(gt_mm, 98)

n_corr = len(CORRECTIONS)
fig, axes = plt.subplots(3, n_corr + 1, figsize=(5*(n_corr+1), 12))

for col, (corr_name, corr_fn) in enumerate(CORRECTIONS.items()):
    img_c = corr_fn(img_rgb)
    depth_rel, t_ms = predict_depth_timed(
        img_c, encoder, depth_decoder, FEED_HEIGHT, FEED_WIDTH, DEVICE)

    valid = (~np.isnan(gt_mm)) & (gt_mm > 0) & (gt_mm < CAP_MM)
    scale = np.median(gt_mm[valid]) / (np.median(depth_rel[valid]) + 1e-8)
    depth_mm = depth_rel * scale

    metrics = compute_all_metrics(img_rgb, img_c, depth_rel, gt_mm, CAP_MM)

    # Fila 0: imagen corregida
    axes[0, col].imshow(img_c)
    axes[0, col].set_title(f"{corr_name}\nPSNR={metrics['PSNR']:.1f} dB  "
                            f"SSIM={metrics['SSIM']:.3f}", fontsize=10)
    axes[0, col].axis("off")

    # Fila 1: depth predicho
    im = axes[1, col].imshow(depth_mm, cmap="magma_r", vmin=0, vmax=vmax)
    axes[1, col].set_title(
        f"AbsRel={metrics['AbsRel']:.4f}\n"
        f"RMSE={metrics['RMSE']:.1f}mm  CD={metrics['Chamfer']:.2f}mm",
        fontsize=9)
    axes[1, col].axis("off")

    # Fila 2: error absoluto por píxel
    err = np.abs(depth_mm - gt_mm)
    err[~valid] = np.nan
    axes[2, col].imshow(err, cmap="hot",
                        vmin=0, vmax=np.nanpercentile(err, 95))
    axes[2, col].set_title(
        f"Error abs — spec={metrics['AbsRel_spec']:.4f}\n"
        f"no-spec={metrics['AbsRel_nospec']:.4f}", fontsize=9)
    axes[2, col].axis("off")

# Columna GT
axes[0, -1].imshow(img_rgb)
axes[0, -1].set_title("Original (referencia)", fontsize=10)
axes[0, -1].axis("off")

axes[1, -1].imshow(gt_mm, cmap="magma_r", vmin=0, vmax=vmax)
axes[1, -1].set_title("GT luz estructurada", fontsize=10)
axes[1, -1].axis("off")

axes[2, -1].imshow(specular_mask(img_rgb), cmap="Reds")
axes[2, -1].set_title("Máscara especular", fontsize=10)
axes[2, -1].axis("off")

plt.colorbar(im, ax=axes[1, :], label="Profundidad (mm)",
             shrink=0.5, pad=0.01)
plt.suptitle(f"Endo-Depth — {EVAL_KEYFRAMES[0][0]}/{EVAL_KEYFRAMES[0][1]}",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(OUT_DIR / "avance3_viz_cualitativa.png",
            dpi=150, bbox_inches="tight")
plt.show()

---
## 10. Performance: tiempo y FPS

Las correcciones de iluminación añaden tiempo de preprocesamiento. Esta sección mide el costo total (corrección + inferencia) para evaluar la viabilidad clínica. En endoscopía en tiempo real, el umbral típico es **≥ 25 FPS** para retroalimentación fluida al cirujano.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

# Medir también el tiempo de la corrección de imagen
print("Midiendo tiempos de preprocesamiento (corrección de imagen)...")
timing_rows = []

for corr_name, corr_fn in CORRECTIONS.items():
    times_corr = []
    for _ in range(10):
        t0 = time.perf_counter()
        _ = corr_fn(img_test)
        times_corr.append((time.perf_counter() - t0) * 1000)

    t_corr = float(np.median(times_corr))

    # Tiempo de inferencia ya medido en df
    t_infer = df[df["Corrección"] == corr_name]["Tiempo_ms"].median()

    timing_rows.append({
        "Corrección":     corr_name,
        "T_corrección_ms": round(t_corr, 1),
        "T_inferencia_ms": round(float(t_infer), 1),
        "T_total_ms":      round(t_corr + float(t_infer), 1),
        "FPS_total":       round(1000 / (t_corr + float(t_infer)), 1),
    })

df_timing = pd.DataFrame(timing_rows)
print(df_timing.to_string(index=False))

# Gráfico de tiempo apilado
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(df_timing))
ax.bar(x, df_timing["T_inferencia_ms"], label="Inferencia",
       color="#2766CB", alpha=0.9)
ax.bar(x, df_timing["T_corrección_ms"], bottom=df_timing["T_inferencia_ms"],
       label="Corrección imagen", color="#E0A800", alpha=0.9)
ax.axhline(1000/25, color="red", linestyle="--", linewidth=1.2,
           label="Umbral 25 FPS (40 ms)")
ax.set_xticks(x)
ax.set_xticklabels(df_timing["Corrección"], fontsize=12)
ax.set_ylabel("Tiempo (ms)")
ax.set_title("Tiempo total: corrección + inferencia Endo-Depth", fontsize=12)
ax.legend(fontsize=9)

for i, row in df_timing.iterrows():
    ax.text(i, row["T_total_ms"] + 0.5,
            f"{row['FPS_total']} FPS", ha="center", fontsize=10, fontweight="bold")

plt.tight_layout()
plt.savefig(OUT_DIR / "avance3_performance.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 11. Exportar resultados

In [ ]:
# Resultados detallados por keyframe
out_csv = OUT_DIR / "avance3_endodepth_results.csv"
df.to_csv(out_csv, index=False)
print(f"Guardado: {out_csv}")

# Resumen por corrección
out_summary = OUT_DIR / "avance3_endodepth_summary.csv"
summary.to_csv(out_summary)
print(f"Guardado: {out_summary}")

# Timing
out_timing = OUT_DIR / "avance3_endodepth_timing.csv"
df_timing.to_csv(out_timing, index=False)
print(f"Guardado: {out_timing}")

print("\n── Resumen final ───────────────────────────────────────")
print(summary[["AbsRel","RMSE","Chamfer","PSNR","SSIM",
               "AbsRel_spec","AbsRel_nospec"]].to_string())

---
## Agradecimientos

El Equipo 52 agradece la orientación del **Dr. Gilberto Ochoa Ruiz** (Tecnológico de Monterrey)
y del **Dr. Ricardo Espinosa Loera** (Universite de Lorraine), cuyas contribuciones
al campo de la visión por computadora y el procesamiento de imágenes médicas
enriquecieron el diseño metodológico de este proyecto.

## Referencias

- Recasens, D., et al. (2021). Endo-Depth-and-Motion: Reconstruction and Tracking in Endoscopic Videos Using Depth Networks and Photometric Constraints. *IEEE RA-L*, 6(4), 7225–7232.
- Godard, C., et al. (2019). Digging Into Self-Supervised Monocular Depth Estimation. *ICCV 2019*.
- Allan, M., et al. (2021). Stereo Correspondence and Reconstruction of Endoscopic Data Challenge. *arXiv:2101.01133*.
- Wang, Z., et al. (2004). Image quality assessment: from error visibility to structural similarity. *IEEE TIP*, 13(4), 600–612.
- Land, E. H., & McCann, J. J. (1971). Lightness and retinex theory. *JOSA*, 61(1), 1–11.